# Bronze layer 

In [2]:
import pandas as pd
import numpy as np
import os
import glob
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/paris_weekends.csv
/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/rome_weekends.csv
/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/berlin_weekdays.csv
/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/amsterdam_weekends.csv
/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/lisbon_weekends.csv
/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/amsterdam_weekdays.csv
/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/rome_weekdays.csv
/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/athens_weekends.csv
/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/budapest_weekends.csv
/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/barcelona_weekdays.csv
/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/barcelona_weekends.csv
/

In [3]:
data_path = '/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities'
all_files = glob.glob(os.path.join(data_path, "*.csv"))
print(f"Number of files: {len(all_files)}")
print(all_files)

Number of files: 20
['/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/paris_weekends.csv', '/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/rome_weekends.csv', '/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/berlin_weekdays.csv', '/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/amsterdam_weekends.csv', '/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/lisbon_weekends.csv', '/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/amsterdam_weekdays.csv', '/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/rome_weekdays.csv', '/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/athens_weekends.csv', '/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/budapest_weekends.csv', '/kaggle/input/datasets/thedevastator/airbnb-prices-in-european-cities/barcelona_weekdays.csv', '/kaggle/input/datasets/thedevastator/airbn

In [4]:
dfs = []

for file in all_files:
    filename = os.path.basename(file).replace('.csv', '')  # e.g. amsterdam_weekdays
    
    if 'weekdays' in filename:
        day_type = 'weekday'
        city = filename.replace('_weekdays', '')
    elif 'weekends' in filename:
        day_type = 'weekend'
        city = filename.replace('_weekends', '')
    else:
        day_type = 'unknown'
        city = filename
    
    temp_df = pd.read_csv(file)
    temp_df['city'] = city.capitalize()
    temp_df['day_type'] = day_type
    dfs.append(temp_df)

df_raw = pd.concat(dfs, ignore_index=True)

In [5]:
os.makedirs('/kaggle/working/bronze', exist_ok=True)
df_raw.to_csv('/kaggle/working/bronze/airbnb_europe_raw.csv', index=False)
print("Bronze Layer Done ")

Bronze Layer Done 


# Silver Layer

# Data Profiling

In [6]:
df_raw.head()

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,0,536.396682,Entire home/apt,False,False,5.0,False,0,1,9.0,...,1.351201,0.212346,390.776775,19.001549,1030.738507,47.550371,2.35900,48.86800,Paris,weekend
1,1,290.101594,Private room,False,True,2.0,True,0,0,10.0,...,0.699821,0.193710,518.478270,25.211044,1218.658866,56.219575,2.35385,48.86282,Paris,weekend
2,2,445.754497,Entire home/apt,False,False,4.0,False,0,1,10.0,...,0.968982,0.294343,432.689942,21.039580,1069.894793,49.356741,2.36023,48.86375,Paris,weekend
3,3,211.343089,Private room,False,True,2.0,False,0,0,10.0,...,3.302319,0.234740,444.555284,21.616533,902.856370,41.650870,2.31714,48.87475,Paris,weekend
4,4,266.334234,Entire home/apt,False,False,2.0,True,0,0,9.0,...,1.402430,0.055052,1013.458689,49.279502,1348.063511,62.189313,2.33408,48.85384,Paris,weekend


In [7]:
print(df_raw.shape)

(51707, 22)


In [8]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  51707 non-null  int64  
 1   realSum                     51707 non-null  float64
 2   room_type                   51707 non-null  object 
 3   room_shared                 51707 non-null  bool   
 4   room_private                51707 non-null  bool   
 5   person_capacity             51707 non-null  float64
 6   host_is_superhost           51707 non-null  bool   
 7   multi                       51707 non-null  int64  
 8   biz                         51707 non-null  int64  
 9   cleanliness_rating          51707 non-null  float64
 10  guest_satisfaction_overall  51707 non-null  float64
 11  bedrooms                    51707 non-null  int64  
 12  dist                        51707 non-null  float64
 13  metro_dist                  517

In [9]:
null_percent = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
print(null_percent[null_percent > 0])

Series([], dtype: float64)


In [10]:
print("Duplicate rows:", df_raw.duplicated().sum())

Duplicate rows: 0


In [11]:
df_raw.describe()

,Unnamed: 0,realSum,person_capacity,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat
count,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.00000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000
mean,1620.502388,279.879591,3.161661,0.291353,0.350204,9.390624,92.628232,1.15876,3.191285,0.681540,294.204105,13.423792,626.856696,22.786177,7.426068,45.671128
std,1217.380366,327.948386,1.298545,0.454390,0.477038,0.954868,8.945531,0.62741,2.393803,0.858023,224.754123,9.807985,497.920226,17.804096,9.799725,5.249263
min,0.000000,34.779339,2.000000,0.000000,0.000000,2.000000,20.000000,0.00000,0.015045,0.002301,15.152201,0.926301,19.576924,0.592757,-9.226340,37.953000
25%,646.000000,148.752174,2.000000,0.000000,0.000000,9.000000,90.000000,1.00000,1.453142,0.248480,136.797385,6.380926,250.854114,8.751480,-0.072500,41.399510
50%,1334.000000,211.343089,3.000000,0.000000,0.000000,10.000000,95.000000,1.00000,2.613538,0.413269,234.331748,11.468305,522.052783,17.542238,4.873000,47.506690
75%,2382.000000,319.694287,4.000000,1.000000,1.000000,10.000000,99.000000,1.00000,4.263077,0.737840,385.756381,17.415082,832.628988,32.964603,13.518825,51.471885
max,5378.000000,18545.450285,6.000000,1.000000,1.000000,10.000000,100.000000,10.00000,25.284557,14.273577,4513.563486,100.000000,6696.156772,100.000000,23.786020,52.641410


In [12]:
!pip install ydata-profiling --quiet
from ydata_profiling import ProfileReport

profile = ProfileReport(df_raw, title="Airbnb Europe Profiling Report", minimal=True)
profile.to_file("/kaggle/working/profiling_report.html")

/tmp/ipykernel_58/2107899743.py:2: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 22/22 [00:00<00:00, 33.23it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
profile.to_notebook_iframe()

In [14]:
print(df_raw.columns.tolist())
df_raw.dtypes

['Unnamed: 0', 'realSum', 'room_type', 'room_shared', 'room_private', 'person_capacity', 'host_is_superhost', 'multi', 'biz', 'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist', 'metro_dist', 'attr_index', 'attr_index_norm', 'rest_index', 'rest_index_norm', 'lng', 'lat', 'city', 'day_type']


Unnamed: 0                      int64
realSum                       float64
room_type                      object
room_shared                      bool
room_private                     bool
person_capacity               float64
host_is_superhost                bool
multi                           int64
biz                             int64
cleanliness_rating            float64
guest_satisfaction_overall    float64
bedrooms                        int64
dist                          float64
metro_dist                    float64
attr_index                    float64
attr_index_norm               float64
rest_index                    float64
rest_index_norm               float64
lng                           float64
lat                           float64
city                           object
day_type                       object
dtype: object

# Data preprocessing

In [15]:
df_clean = df_raw.copy()
df_clean = df_clean.drop(columns=['Unnamed: 0'])
df_clean['city'] = df_clean['city'].astype('category')
df_clean['day_type'] = df_clean['day_type'].astype('category')
df_clean['room_type'] = df_clean['room_type'].astype('category')
df_clean['multi'] = df_clean['multi'].astype(bool)
df_clean['biz'] = df_clean['biz'].astype(bool)

In [16]:
before = len(df_clean)
df_clean = df_clean[df_clean['realSum'] > 0]
df_clean = df_clean[df_clean['person_capacity'] > 0]
print(f"removed {before - len(df_clean)} rows due to presence of outlair")

Q1 = df_clean['realSum'].quantile(0.25)
Q3 = df_clean['realSum'].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

before = len(df_clean)
df_clean = df_clean[(df_clean['realSum'] >= max(0, lower_limit)) & (df_clean['realSum'] <= upper_limit)]
print(f"removed {before - len(df_clean)} row due to presence of outlier (realSum)")

print("\nShape After cleaning:", df_clean.shape)
df_clean.dtypes

removed 0 rows due to presence of outlair
removed 3662 row due to presence of outlier (realSum)

Shape After cleaning: (48045, 21)


realSum                        float64
room_type                     category
room_shared                       bool
room_private                      bool
person_capacity                float64
host_is_superhost                 bool
multi                             bool
biz                               bool
cleanliness_rating             float64
guest_satisfaction_overall     float64
bedrooms                         int64
dist                           float64
metro_dist                     float64
attr_index                     float64
attr_index_norm                float64
rest_index                     float64
rest_index_norm                float64
lng                            float64
lat                            float64
city                          category
day_type                      category
dtype: object

In [17]:
df_clean['realSum'].describe()

count    48045.000000
mean       228.980509
std        112.045344
min         34.779339
25%        144.483670
50%        201.529002
75%        287.992495
max        576.008948
Name: realSum, dtype: float64

In [18]:
os.makedirs('/kaggle/working/silver', exist_ok=True)
df_clean.to_parquet('/kaggle/working/silver/airbnb_europe_clean.parquet', index=False)
df_clean.to_csv('/kaggle/working/silver/airbnb_europe_clean.csv', index=False)
print("Silver Layer Done")

Silver Layer Done


# Web Scrapping Using Selinium

# Set up 

In [19]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 124 not upgraded.
Need to get 11.2 MB/145 MB of archives.
After this operation, 487 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 liba

In [20]:
!google-chrome --version

Google Chrome 150.0.7871.114 


In [21]:
!pip install selenium webdriver-manager --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 67.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [22]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.binary_location = '/usr/bin/google-chrome'

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

driver.get("https://www.google.com")
print(driver.title)
driver.quit()

Google


In [23]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def get_listing_urls(search_url, max_listings=20):
    
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.binary_location = '/usr/bin/google-chrome'
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

    driver.get(search_url)
    time.sleep(4)

    
    links = driver.find_elements(By.XPATH, "//a[contains(@href,'/rooms/')]")
    urls = []
    for link in links:
        href = link.get_attribute('href')
        if href and '/rooms/' in href:
            clean_url = href.split('?')[0]  
            if clean_url not in urls:
                urls.append(clean_url)

    driver.quit()
    return urls[:max_listings]

search_pages = [
    'https://www.airbnb.com/s/Paris--France/homes',
    'https://www.airbnb.com/s/Berlin--Germany/homes',
    'https://www.airbnb.com/s/Rome--Italy/homes',
    'https://www.airbnb.com/s/Vienna--Austria/homes'
]

all_listing_urls = []
for page in search_pages:
    print(f"Getting links from: {page}")
    urls = get_listing_urls(page, max_listings=10)
    print(f"  I found {len(urls)} Advertisement")
    all_listing_urls.extend(urls)

print(f"\nTotal links: {len(all_listing_urls)}")
print(all_listing_urls[:5])  

Getting links from: https://www.airbnb.com/s/Paris--France/homes
  I found 10 Advertisement
Getting links from: https://www.airbnb.com/s/Berlin--Germany/homes
  I found 10 Advertisement
Getting links from: https://www.airbnb.com/s/Rome--Italy/homes
  I found 10 Advertisement
Getting links from: https://www.airbnb.com/s/Vienna--Austria/homes
  I found 10 Advertisement

Total links: 40
['https://www.airbnb.com/rooms/1156424874708107231', 'https://www.airbnb.com/rooms/1173047051903832695', 'https://www.airbnb.com/rooms/1619572243373657992', 'https://www.airbnb.com/rooms/1330790437011371935', 'https://www.airbnb.com/rooms/1191324350983013220']


In [24]:
listing_urls = [
    'https://www.airbnb.com/s/Paris--France/homes',
    'https://www.airbnb.com/s/Berlin--Germany/homes',
]

In [25]:
listing_urls = all_listing_urls
print(listing_urls)

['https://www.airbnb.com/rooms/1156424874708107231', 'https://www.airbnb.com/rooms/1173047051903832695', 'https://www.airbnb.com/rooms/1619572243373657992', 'https://www.airbnb.com/rooms/1330790437011371935', 'https://www.airbnb.com/rooms/1191324350983013220', 'https://www.airbnb.com/rooms/1012178000290782670', 'https://www.airbnb.com/rooms/1265039206175532513', 'https://www.airbnb.com/rooms/1265036898105754633', 'https://www.airbnb.com/rooms/1334829999547142910', 'https://www.airbnb.com/rooms/1265037710933744777', 'https://www.airbnb.com/rooms/721527807549293451', 'https://www.airbnb.com/rooms/798462813111327157', 'https://www.airbnb.com/rooms/1182435035023787880', 'https://www.airbnb.com/rooms/917109042412215124', 'https://www.airbnb.com/rooms/1348574879310010332', 'https://www.airbnb.com/rooms/48651189', 'https://www.airbnb.com/rooms/1461489656921679773', 'https://www.airbnb.com/rooms/1679017853409132652', 'https://www.airbnb.com/rooms/1624772462365905516', 'https://www.airbnb.com/r

In [26]:
import time
import random
import re
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd


def get_details_and_fav(driver, wait):
    rating, reviews, is_fav = "N/A", "0", 0

    try:
        try:
            review_link = wait.until(EC.presence_of_element_located(
                (By.XPATH, "//a[contains(@href, 'reviews')] | //button[contains(., 'reviews')]")
            ))
        except Exception:
            try:
                h1_parent = driver.find_element(By.TAG_NAME, "h1").find_element(By.XPATH, "./..").text
                if "New" in h1_parent:
                    return "New", "0", 0
            except Exception:
                pass
            return rating, reviews, is_fav

        aria_text = review_link.get_attribute("aria-label") or ""
        link_text = review_link.get_attribute("textContent").strip()
        clean_link = " ".join(link_text.split())

        parent_text = review_link.find_element(By.XPATH, "./..").get_attribute("textContent").strip()
        clean_parent = " ".join(parent_text.split())
        text_to_search = aria_text if aria_text else (clean_parent if clean_parent else clean_link)

        is_fav = 1 if ("Guest favorite" in aria_text or "Guest favorite" in clean_link
                       or "Guest favorite" in clean_parent) else 0

        if "New" in text_to_search and not re.search(r"\d\.\d", text_to_search):
            rating = "New"
        else:
            r_match = re.search(r"(\d\.\d{1,2})", text_to_search)
            if r_match:
                rating = r_match.group(1)
                text_to_search = text_to_search[:r_match.start()] + " " + text_to_search[r_match.end():]

        rev_match = re.search(r"(\d+)\s*[Rr]eview", text_to_search)
        if rev_match:
            reviews = rev_match.group(1)

    except Exception:
        pass

    return rating, reviews, is_fav


def scrape_one_listing(driver, wait, url):
    driver.get(url)
    time.sleep(3)

    try:
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
    except Exception:
        pass

    try:
        name = driver.find_element(By.TAG_NAME, "h1").text
    except Exception:
        name = "No Title"

    is_superhost = "No"
    try:
        driver.execute_script("window.scrollBy(0, 500);")
        time.sleep(0.5)
        body_text = driver.find_element(By.TAG_NAME, "body").text
        if "Superhost" in body_text:
            is_superhost = "Yes"
    except Exception:
        pass

    price = "N/A"
    try:
        price = driver.find_element(
            By.XPATH,
            "//div[@data-plugin-in-point-id='BOOK_IT_SIDEBAR']//button//span"
        ).text
    except Exception:
        pass

    rating, reviews, is_fav = get_details_and_fav(driver, wait)

    return {
        "Name": name,
        "Price": price,
        "Superhost": is_superhost,
        "is_fav": is_fav,
        "Rating": rating,
        "Reviews": reviews,
        "URL": url,
    }

chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1920,1080")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
wait = WebDriverWait(driver, 8)

results = []
for i, url in enumerate(all_listing_urls, 1):
    try:
        print(f"[{i}/{len(all_listing_urls)}] Scraping: {url}")
        data = scrape_one_listing(driver, wait, url)
        results.append(data)
    except Exception as e:
        print(f"Failed in {url}: {e}")
    time.sleep(random.uniform(2, 4))

driver.quit()

df_scraped = pd.DataFrame(results)

os.makedirs('/kaggle/working/bronze', exist_ok=True)
df_scraped.to_csv('/kaggle/working/bronze/airbnb_scraped_features.csv', index=False)
print("Saved sucessfully")
df_scraped

[1/40] Scraping: https://www.airbnb.com/rooms/1156424874708107231
[2/40] Scraping: https://www.airbnb.com/rooms/1173047051903832695
[3/40] Scraping: https://www.airbnb.com/rooms/1619572243373657992
[4/40] Scraping: https://www.airbnb.com/rooms/1330790437011371935
[5/40] Scraping: https://www.airbnb.com/rooms/1191324350983013220
[6/40] Scraping: https://www.airbnb.com/rooms/1012178000290782670
[7/40] Scraping: https://www.airbnb.com/rooms/1265039206175532513
[8/40] Scraping: https://www.airbnb.com/rooms/1265036898105754633
[9/40] Scraping: https://www.airbnb.com/rooms/1334829999547142910
[10/40] Scraping: https://www.airbnb.com/rooms/1265037710933744777
[11/40] Scraping: https://www.airbnb.com/rooms/721527807549293451
[12/40] Scraping: https://www.airbnb.com/rooms/798462813111327157
[13/40] Scraping: https://www.airbnb.com/rooms/1182435035023787880
[14/40] Scraping: https://www.airbnb.com/rooms/917109042412215124
[15/40] Scraping: https://www.airbnb.com/rooms/1348574879310010332
[16/40]

,Name,Price,Superhost,is_fav,Rating,Reviews,URL
0,Chic Family Apartment with AC in a Prime Location,Check availability,No,1,5.0,013,https://www.airbnb.com/rooms/1156424874708107231
1,Best Eiffel Tower View: Refurbished Studio !,Check availability,Yes,1,4.97,9772,https://www.airbnb.com/rooms/1173047051903832695
2,Charming apartment - 4P- 1BR -Arc de Triomphe,Check availability,Yes,1,4.95,9519,https://www.airbnb.com/rooms/1619572243373657992
3,Elegant apartment close to the Champs-Élysées,Check availability,Yes,1,5.0,033,https://www.airbnb.com/rooms/1330790437011371935
4,Madeleine Saint Honoré - The Madeleine Hut,Check availability,Yes,1,4.99,99121,https://www.airbnb.com/rooms/1191324350983013220
5,"300m from Notre Dame, 80m2 with A/C and balcony",Check availability,Yes,0,5.0,2,https://www.airbnb.com/rooms/1012178000290782670
6,Hotel Astoria - Astotel,Check availability,No,0,4.73,301,https://www.airbnb.com/rooms/1265039206175532513
7,Hotel 34B - Astotel,Check availability,No,0,4.78,331,https://www.airbnb.com/rooms/1265036898105754633
8,West Paris Suite,Check availability,No,0,4.83,24,https://www.airbnb.com/rooms/1334829999547142910
9,Hotel Monterosa - Astotel,Check availability,No,0,4.79,551,https://www.airbnb.com/rooms/1265037710933744777


In [27]:
df_scraped.to_csv('/kaggle/working/scraped_data.csv', index=False)

print("Web scrabed file saved successfully")

Web scrabed file saved successfully


In [28]:
df_scraped = pd.read_csv('/kaggle/working/scraped_data.csv')

print("columns of cleaned data:", df_clean.columns.tolist())
print("columns of new scrabed data:", df_scraped.columns.tolist())

columns of cleaned data: ['realSum', 'room_type', 'room_shared', 'room_private', 'person_capacity', 'host_is_superhost', 'multi', 'biz', 'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist', 'metro_dist', 'attr_index', 'attr_index_norm', 'rest_index', 'rest_index_norm', 'lng', 'lat', 'city', 'day_type']
columns of new scrabed data: ['Name', 'Price', 'Superhost', 'is_fav', 'Rating', 'Reviews', 'URL']


# Merging cleaned data with scrabed data

In [29]:
mapping = {
    'Price': 'realSum',
    'Superhost': 'host_is_superhost',
    'Rating': 'guest_satisfaction_overall'
}
df_scraped_renamed = df_scraped.rename(columns=mapping)

In [30]:
import pandas as pd
import numpy as np

# 1. Filter the scraped dataset to take only the three matching columns to avoid duplication or extra columns
scraped_matching_columns = ['realSum', 'host_is_superhost', 'guest_satisfaction_overall']
df_scraped_filtered = df_scraped_renamed[scraped_matching_columns]

# 2. Concatenate while keeping all the original 21 columns of df_clean intact
df_final_perfect = pd.concat([df_clean, df_scraped_filtered], ignore_index=True)

# 3. Handle data types, convert ratings to numeric, and convert text errors (like 'New') into NaN
df_final_perfect['guest_satisfaction_overall'] = pd.to_numeric(
    df_final_perfect['guest_satisfaction_overall'], 
    errors='coerce'
)

# 4. Unify the scale: If any rating is out of 5 (<= 5.0), multiply it by 20 to make it out of 100
df_final_perfect['guest_satisfaction_overall'] = df_final_perfect['guest_satisfaction_overall'].apply(
    lambda x: x * 20 if (pd.notnull(x) and x <= 5.0) else x
)

# 5. Calculate the median safely and fill any remaining missing values in the rating column
median_val = df_final_perfect['guest_satisfaction_overall'].median()
df_final_perfect['guest_satisfaction_overall'] = df_final_perfect['guest_satisfaction_overall'].fillna(median_val)


# === Verification and Display ===
print("=== 1) All Columns in the Final Dataset ===")
print(df_final_perfect.columns.tolist())
print(f"\nTotal number of columns: {len(df_final_perfect.columns)} (Should be the original 21 columns)")

print("\n=== 2) Final Dataset Shape ===")
print("Shape:", df_final_perfect.shape)

print("\n=== 3) Preview of First 5 Rows ===")
display(df_final_perfect.head())


# === 6. Export the full dataset with all columns to an Excel file ===
excel_filename = '/kaggle/working/perfect_airbnb_combined_data.xlsx'
df_final_perfect.to_excel(excel_filename, index=False)

print(f"\n=== Success! Full Excel file saved as: {excel_filename} ===")
print("Go to your Output folder, click refresh (🔄), and download the new file!")

=== 1) All Columns in the Final Dataset ===
['realSum', 'room_type', 'room_shared', 'room_private', 'person_capacity', 'host_is_superhost', 'multi', 'biz', 'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist', 'metro_dist', 'attr_index', 'attr_index_norm', 'rest_index', 'rest_index_norm', 'lng', 'lat', 'city', 'day_type']

Total number of columns: 21 (Should be the original 21 columns)

=== 2) Final Dataset Shape ===
Shape: (48085, 21)

=== 3) Preview of First 5 Rows ===


,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,...,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,536.396682,Entire home/apt,False,False,5.0,False,False,True,9.0,89.0,...,1.351201,0.212346,390.776775,19.001549,1030.738507,47.550371,2.35900,48.86800,Paris,weekend
1,290.101594,Private room,False,True,2.0,True,False,False,10.0,97.0,...,0.699821,0.193710,518.478270,25.211044,1218.658866,56.219575,2.35385,48.86282,Paris,weekend
2,445.754497,Entire home/apt,False,False,4.0,False,False,True,10.0,100.0,...,0.968982,0.294343,432.689942,21.039580,1069.894793,49.356741,2.36023,48.86375,Paris,weekend
3,211.343089,Private room,False,True,2.0,False,False,False,10.0,94.0,...,3.302319,0.234740,444.555284,21.616533,902.856370,41.650870,2.31714,48.87475,Paris,weekend
4,266.334234,Entire home/apt,False,False,2.0,True,False,False,9.0,88.0,...,1.402430,0.055052,1013.458689,49.279502,1348.063511,62.189313,2.33408,48.85384,Paris,weekend



=== Success! Full Excel file saved as: /kaggle/working/perfect_airbnb_combined_data.xlsx ===
Go to your Output folder, click refresh (🔄), and download the new file!


In [31]:
df = df_clean.copy()

# ---------- Dim City ----------
dim_city = df[['city']].drop_duplicates().reset_index(drop=True)
dim_city['city_key'] = dim_city.index + 1

# ---------- Dim Room Type ----------
dim_room_type = df[['room_type', 'room_shared', 'room_private']].drop_duplicates().reset_index(drop=True)
dim_room_type['room_type_key'] = dim_room_type.index + 1

# ---------- Dim Host ----------
dim_host = df[['host_is_superhost', 'multi', 'biz']].drop_duplicates().reset_index(drop=True)
dim_host['host_key'] = dim_host.index + 1

# ---------- Dim Day Type ----------
dim_day_type = df[['day_type']].drop_duplicates().reset_index(drop=True)
dim_day_type['day_type_key'] = dim_day_type.index + 1

# ---------- Fact Table ----------
fact = df.merge(dim_city, on='city') \
         .merge(dim_room_type, on=['room_type', 'room_shared', 'room_private']) \
         .merge(dim_host, on=['host_is_superhost', 'multi', 'biz']) \
         .merge(dim_day_type, on='day_type')

fact_listing = fact[[
    'city_key', 'room_type_key', 'host_key', 'day_type_key',
    'realSum', 'person_capacity', 'cleanliness_rating',
    'guest_satisfaction_overall', 'bedrooms', 'dist', 'metro_dist',
    'attr_index', 'attr_index_norm', 'rest_index', 'rest_index_norm',
    'lat', 'lng'
]].rename(columns={'realSum': 'price'})

fact_listing.insert(0, 'listing_key', range(1, len(fact_listing) + 1))

print("Dataframes ready ✅")
print(dim_city.shape, dim_room_type.shape, dim_host.shape, dim_day_type.shape, fact_listing.shape)

Dataframes ready ✅
(10, 2) (3, 4) (6, 4) (2, 2) (48045, 18)


In [32]:
import os
os.makedirs('/kaggle/working/gold_csv', exist_ok=True)

dim_city.to_csv('/kaggle/working/gold_csv/dim_city.csv', index=False)
dim_room_type.to_csv('/kaggle/working/gold_csv/dim_room_type.csv', index=False)
dim_host.to_csv('/kaggle/working/gold_csv/dim_host.csv', index=False)
dim_day_type.to_csv('/kaggle/working/gold_csv/dim_day_type.csv', index=False)
fact_listing.to_csv('/kaggle/working/gold_csv/fact_listing.csv', index=False)

print("CSV files ready in /kaggle/working/gold_csv")

CSV files ready in /kaggle/working/gold_csv
